In [35]:
import sys
import os
# Get the current directory and parent directory
current_dir = os.getcwd()
parent_dir = os.path.dirname(current_dir)
sys.path.insert(0, parent_dir)
sys.path.insert(0, current_dir)

# Import from the-machine module
import importlib.util
the_machine_path = os.path.join(parent_dir, "the-machine")
sys.path.insert(0, the_machine_path)

# Import all from the-machine and custom_matroid_functions
from HM_module import *
from custom_matroid_functions import *

In [36]:
P = LaurentPolynomialRing(ZZ, 'x')
x = P.gen()
r, n = 3, 3  # rank r, field order n
M = matroids.PG(r - 1, n)  # Projective geometry matroid PG(r-1, n) of rank r
L = M.lattice_of_flats()
r = M.rank()
rank_func = lambda x : M.rank(x)

#e = next(x for x in M.groundset() if x  not in M.coloops())
e = list(M.groundset())[0]
print(M.groundset())
print("deleting ", e)
mdele = M.delete(e)
Eq_M = create_deformed_algebra(M, base_ring=P)
Eq_mdele = create_deformed_algebra(mdele, base_ring=P)
E_mdele = Eq_mdele.natural()

E = Eq_M.natural()
E_basis = E.basis()
E_mdele_basis = E_mdele.basis()

C = Eq_M.C()
C_dele = Eq_mdele.C()
C_basis = C.basis()
C_dele_basis = C_dele.basis()

flats_keys = list(E_basis.keys())

chow = chow_polynomial
chibar = reduced_characteristic_polynomial
char_poly = characteristic_polynomial
kl_poly = lambda M: M.lattice_of_flats().kazhdan_lusztig_polynomial()(x)
grev = lambda M: P(x**M.rank())
mu_rev = lambda M: P(x**M.rank() * char_poly(M)(0))
mu = lambda M: P(char_poly(M)(0))

print(chibar(M))
print(mu_rev(M))
print(kl_poly(M))

frozenset({0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12})
deleting  0
x^2 - 12*x + 27
-27*x^3
1


In [37]:
def flat_division(M, e):
    L = M.lattice_of_flats()
    flat_div = {'S':set(), 'T':set(), 'A':set(), 'B':set()}
    for F in L:
        if e in F:
            if F - {e} in L:
                flat_div['T'].add(F)
            else:
                flat_div['A'].add(F)
        else:
            if F.union({e}) in L:
                flat_div['S'].add(F)
            else:
                flat_div['B'].add(F)
    return flat_div

def get_pal0_part(f):
    """
    Extracts the Pal(0) part of a Laurent polynomial f.
    """
    if f == 0: return f
    x_var = f.parent().gen()
    f_inv = f.subs({x_var: x_var**-1})
    numerator = (x_var**2) * f_inv - f
    denominator = (x_var**2) - 1
    q, r = numerator.quo_rem(denominator)
    return q


flat_div = flat_division(M, e)
#flat_div['S'].remove(frozenset())

def convolve(f, g):
    return lambda M: sum([f(M.delete(M.groundset() - F))(x) * g(M.contract(F))(x) for F in list(M.lattice_of_flats())])

conv = lambda f, g: convolve(f, g)(M)

def rev(f):
    return lambda M: f(M)(x**-1) * x**M.rank()

rightAugpoly = convolve(chow, rev(kl_poly))
bbasepoly = convolve(mu_rev, rightAugpoly)
zeta = lambda M: P(1)
klrev = rev(kl_poly)
leftAugpoly = convolve(grev, chow_polynomial)

In [38]:
def get_matroid_key(M):
    rank = M.rank()
    size = len(M.groundset())
    try:
        can = M.canonical_form()
        return (rank, size, str(can))
    except AttributeError:
        bases = tuple(sorted(tuple(sorted(b)) for b in M.bases()))
        return (rank, size, str(bases))

def convolution_inverse(f):
    cache_inv = {}
    
    def f_inv(M):
        m_key = get_matroid_key(M)
        if m_key in cache_inv:
            return cache_inv[m_key]
        
        if M.rank() == 0 and M.size() == 0:
            res = 1 / f(M)
            cache_inv[m_key] = res
            return res
        
        # f_inv(M) = - (1/f(empty)) * sum_{empty != F <= E} f(M|F) f_inv(M/F)
        empty_matroid = matroids.Uniform(0, 0)
        f0 = f(empty_matroid)
        
        L = M.lattice_of_flats()
        E = M.groundset()
        
        total = P(0)
        for F in L:
            if len(F) == 0: continue
            restriction = M.delete(list(set(E) - set(F)))
            contraction = M.contract(list(F))
            total += f(restriction) * f_inv(contraction)
            
        res = -total / f0
        cache_inv[m_key] = res
        return res
    
    return f_inv

In [39]:
print("rank of M: ", r)
print("chow polynomial of M: ", chow_polynomial(M))
print(reduced_characteristic_polynomial(M))
for F in flat_div['S']:
    print(F)

rank of M:  3
chow polynomial of M:  x^2 + 14*x + 1
x^2 - 12*x + 27
frozenset()


In [40]:
def deleconj(M, e, f):
    mdele = M.delete(e)
    og_matroid_poly = f(M)
    deleted_matroid_poly = f(mdele)
    con_matroid_poly = f(M.contract(e))
    sum2 = 0
    for F in flat_div['S']:
        poly1 = f(M.contract(F.union({e})))
        poly2 = chow_polynomial(M.delete(M.groundset() - F))
        term = (poly1) * poly2
        sum2 += term
        print(F)
        print("poly1:   ", poly1)
        print("poly2:   ", poly2)
        print("term: ", term)
        print("--------------------------------")
        print("********************************")
    print("summation equals:", sum2)
    print("f(M):     ", og_matroid_poly)
    print("f(M\\e):  ", deleted_matroid_poly)
    print("f(M/e):  ", con_matroid_poly)
    return P(deleted_matroid_poly - con_matroid_poly + sum2)

right = deleconj(M, e, rightAugpoly)
print("right", right)

frozenset()
poly1:    1 + 5*x + x^2
poly2:    1
term:  1 + 5*x + x^2
--------------------------------
********************************
summation equals: 1 + 5*x + x^2
f(M):      1 + 27*x + 27*x^2 + x^3
f(M\e):   1 + 27*x + 27*x^2 + x^3
f(M/e):   1 + 5*x + x^2
right 1 + 27*x + 27*x^2 + x^3
